In [1]:
# Cell 1: imports + paths
from pathlib import Path
from collections import defaultdict
import json, re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd() / "accessops_coco_ai"
DATA_DIR = PROJECT_ROOT / "data" / "coco"
RAW_DIR = DATA_DIR / "raw"
ANN_DIR = DATA_DIR / "annotations"
TRAIN_IMG_DIR = DATA_DIR / "train2017"
VAL_IMG_DIR = DATA_DIR / "val2017"

ART_DIR = PROJECT_ROOT / "artifacts"
SPLITS_DIR = ART_DIR / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

for p in [RAW_DIR, ANN_DIR, TRAIN_IMG_DIR, VAL_IMG_DIR, ART_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /teamspace/studios/this_studio/accessops_coco_ai/notebooks/accessops_coco_ai


In [2]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

2026-04-06 17:02:14.398401: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-06 17:02:14.421052: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-06 17:02:14.449848: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-06 17:02:14.449897: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-06 17:02:14.470275: I tensorflow/core/platform/cpu_feature_gua

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# Cell 2: download COCO 2017 files (full) if missing
import subprocess

downloads = [
    ("http://images.cocodataset.org/zips/train2017.zip", RAW_DIR / "train2017.zip"),
    ("http://images.cocodataset.org/zips/val2017.zip", RAW_DIR / "val2017.zip"),
    ("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", RAW_DIR / "annotations_trainval2017.zip"),
]

for url, out in downloads:
    if out.exists() and out.stat().st_size > 0:
        print("Exists:", out.name)
        continue
    print("Downloading:", out.name)
    subprocess.run(["wget", "-c", url, "-O", str(out)], check=True)

print("Download step complete.")


Exists: train2017.zip
Exists: val2017.zip
Exists: annotations_trainval2017.zip
Download step complete.


In [4]:
# Cell 3: unzip if needed
import zipfile

to_extract = [
    (RAW_DIR / "train2017.zip", DATA_DIR),
    (RAW_DIR / "val2017.zip", DATA_DIR),
    (RAW_DIR / "annotations_trainval2017.zip", DATA_DIR),
]

for zpath, dest in to_extract:
    print("Extracting:", zpath.name)
    with zipfile.ZipFile(zpath, "r") as zf:
        zf.extractall(dest)

print("Extraction complete.")
print("train json:", (ANN_DIR / "captions_train2017.json").exists())
print("val json:", (ANN_DIR / "captions_val2017.json").exists())


Extracting: train2017.zip
Extracting: val2017.zip
Extracting: annotations_trainval2017.zip
Extraction complete.
train json: True
val json: True


In [5]:
# Cell 4: build train/val caption rows from COCO JSON
def clean_text(x: str) -> str:
    x = str(x).lower().strip()
    x = re.sub(r"[^a-z0-9\s]", "", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def coco_to_df(captions_json_path: Path, split_name: str, image_dir: Path) -> pd.DataFrame:
    with open(captions_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    id_to_file = {img["id"]: img["file_name"] for img in data["images"]}
    counter = defaultdict(int)
    rows = []

    for ann in tqdm(data["annotations"], desc=f"rows:{split_name}"):
        img_name = id_to_file[ann["image_id"]]
        counter[img_name] += 1
        rows.append({
            "image_name": img_name,
            "image_path": str((image_dir / img_name).resolve()),
            "comment_number": int(counter[img_name]),
            "comment": ann["caption"],
            "comment_clean": clean_text(ann["caption"]),
            "split": split_name
        })
    return pd.DataFrame(rows)

train_df = coco_to_df(ANN_DIR / "captions_train2017.json", "train", TRAIN_IMG_DIR)
val_all_df = coco_to_df(ANN_DIR / "captions_val2017.json", "val", VAL_IMG_DIR)

print("train rows:", len(train_df))
print("val rows:", len(val_all_df))


rows:train:   0%|          | 0/591753 [00:00<?, ?it/s]

rows:val:   0%|          | 0/25014 [00:00<?, ?it/s]

train rows: 591753
val rows: 25014


In [6]:
# Cell 5: split val2017 into val/test by image (50/50 deterministic)
val_image_names = sorted(val_all_df["image_name"].unique())
mid = len(val_image_names) // 2

val_keep = set(val_image_names[:mid])
test_keep = set(val_image_names[mid:])

val_df = val_all_df[val_all_df["image_name"].isin(val_keep)].copy()
test_df = val_all_df[val_all_df["image_name"].isin(test_keep)].copy()
test_df["split"] = "test"

full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

# keep only rows whose image file exists
exists_mask = full_df["image_path"].map(lambda p: Path(p).exists())
full_df = full_df[exists_mask].reset_index(drop=True)

print(full_df["split"].value_counts())
print(full_df.groupby("split")["image_name"].nunique())


split
train    591753
val       12508
test      12506
Name: count, dtype: int64
split
test       2500
train    118287
val        2500
Name: image_name, dtype: int64


In [7]:
# Cell 6: save stage outputs
csv_out = ART_DIR / "captions_clean_with_splits.csv"
full_df.to_csv(csv_out, index=False)

train_images = sorted(full_df.loc[full_df["split"] == "train", "image_name"].unique())
val_images = sorted(full_df.loc[full_df["split"] == "val", "image_name"].unique())
test_images = sorted(full_df.loc[full_df["split"] == "test", "image_name"].unique())

(SPLITS_DIR / "train_images.txt").write_text("\n".join(train_images), encoding="utf-8")
(SPLITS_DIR / "val_images.txt").write_text("\n".join(val_images), encoding="utf-8")
(SPLITS_DIR / "test_images.txt").write_text("\n".join(test_images), encoding="utf-8")

print("Saved:", csv_out)
print("Saved split text files in:", SPLITS_DIR)


Saved: /teamspace/studios/this_studio/accessops_coco_ai/notebooks/accessops_coco_ai/artifacts/captions_clean_with_splits.csv
Saved split text files in: /teamspace/studios/this_studio/accessops_coco_ai/notebooks/accessops_coco_ai/artifacts/splits


In [8]:
# Cell 7: sanity metrics + report
caption_len = full_df["comment_clean"].str.split().str.len()

report = {
    "dataset": "MS COCO 2017",
    "rows_total": int(len(full_df)),
    "rows_by_split": {k: int(v) for k, v in full_df["split"].value_counts().to_dict().items()},
    "unique_images_by_split": {
        "train": int(len(train_images)),
        "val": int(len(val_images)),
        "test": int(len(test_images)),
    },
    "missing_comment_clean": int(full_df["comment_clean"].isna().sum()),
    "empty_comment_clean": int((full_df["comment_clean"].str.len() == 0).sum()),
    "caption_len_words": {
        "min": int(caption_len.min()),
        "max": int(caption_len.max()),
        "mean": float(caption_len.mean()),
        "p95": float(caption_len.quantile(0.95)),
    },
    "columns": list(full_df.columns),
}

(ART_DIR / "stage1_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))


{
  "dataset": "MS COCO 2017",
  "rows_total": 616767,
  "rows_by_split": {
    "train": 591753,
    "val": 12508,
    "test": 12506
  },
  "unique_images_by_split": {
    "train": 118287,
    "val": 2500,
    "test": 2500
  },
  "missing_comment_clean": 0,
  "empty_comment_clean": 0,
  "caption_len_words": {
    "min": 5,
    "max": 49,
    "mean": 10.454249335648631,
    "p95": 15.0
  },
  "columns": [
    "image_name",
    "image_path",
    "comment_number",
    "comment",
    "comment_clean",
    "split"
  ]
}


In [9]:
# Cell 8: gate check
required = [
    ART_DIR / "captions_clean_with_splits.csv",
    ART_DIR / "stage1_report.json",
    SPLITS_DIR / "train_images.txt",
    SPLITS_DIR / "val_images.txt",
    SPLITS_DIR / "test_images.txt",
]
for p in required:
    assert p.exists(), f"Missing: {p}"

print("STAGE 1 PASS")


STAGE 1 PASS
